<a href="https://colab.research.google.com/github/JCARNEIROX/IA367-Aprendizado-Reforco/blob/main/Lista01/Lista1_Ex8_RA239738_RA167097.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Convolutional neural networks

O objetivo desse código é testar diferentes configurações de CNN aplicadass ao dataset MNIST.

O conjunto de dados MNIST é uma grande coleção de imagens de dígitos manuscritos (de 0 a 9) usada para treinar e testar sistemas de aprendizado de máquina, especialmente para classificação de imagens.

## Training a CNN on MNIST

In [ ]:
#Import das bibliotecas necessárias

import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf

print(tf.__version__)

We import and normalize the MNIST data like last time, except we do not reshape the images: they stay with the shape (28, 28, 1):

In [ ]:
# Carrega o conjunto de dados MNIST, que contém imagens de dígitos escritos à mão.
# O método 'load_data()' divide os dados em conjuntos de treinamento (X_train, t_train) e de teste (X_test, t_test).
# X_train e X_test contêm as imagens, enquanto t_train e t_test contêm os rótulos (os dígitos de 0 a 9).
(X_train, t_train), (X_test, t_test) = tf.keras.datasets.mnist.load_data()

# Imprime as dimensões (shape) dos dados de treinamento para verificar o formato.
# O resultado mostrará (número de imagens, altura, largura).
print("Training data:", X_train.shape, t_train.shape)

# Imprime as dimensões (shape) dos dados de teste.
print("Test data:", X_test.shape, t_test.shape)


# Normaliza os valores dos pixels das imagens.
X_train = X_train.reshape(-1, 28, 28, 1).astype('float32') / 255.
X_test = X_test.reshape(-1, 28, 28, 1).astype('float32') / 255.


# Realiza a remoção da média (mean removal), uma técnica de pré-processamento.
# 1. 'np.mean(X_train, axis=0)': Calcula a imagem média de todo o conjunto de treinamento.
#    'axis=0' especifica que a média deve ser calculada ao longo do eixo das amostras (imagens).
X_mean = np.mean(X_train, axis=0)
# 2. 'X_train -= X_mean': Subtrai a imagem média de cada imagem no conjunto de treinamento.
#    Isso centraliza os dados em torno de zero, o que pode ajudar na convergência do modelo.
X_train -= X_mean
# 3. Aplica a mesma transformação ao conjunto de teste usando a média calculada a partir do conjunto de treinamento.
X_test -= X_mean


# Converte os rótulos (labels) numéricos para o formato one-hot encoding.
# O '10' indica o número total de classes (dígitos de 0 a 9).
T_train = tf.keras.utils.to_categorical(t_train, 10)
T_test = tf.keras.utils.to_categorical(t_test, 10)

We can now define the CNN defined in the first image:

* a convolutional layer with 16 3x3 filters, using valid padding and ReLU transfer functions,
* a max-pooling layer over 2x2 regions,
* a fully-connected layer with 100 ReLU neurons,
* a softmax layer with 10 neurons.

The CNN will be trained on MNIST using SGD with momentum.

The following code defines this basic network in keras:

In [ ]:
# Limpa a sessão backend do Keras. Isso remove quaisquer modelos ou grafos computacionais
# criados anteriormente, liberando memória e evitando conflitos.
tf.keras.backend.clear_session()

model = tf.keras.models.Sequential()

# Adiciona a camada de entrada. 'shape=(28, 28, 1)' define o formato dos dados de entrada:
# imagens de 28x28 pixels com 1 canal de cor (escala de cinza).
model.add(tf.keras.layers.Input(shape=(28, 28, 1)))


# Adiciona a primeira camada convolucional (Conv2D).
# '16' é o número de filtros (ou mapas de características) que a camada aprenderá.
# '(3, 3)' é o tamanho do kernel (filtro) da convolução.
# 'padding='valid'' significa que não será aplicado preenchimento à imagem de entrada.
model.add(tf.keras.layers.Conv2D(16, (3, 3), padding='valid'))
# Adiciona uma função de ativação 'relu' (Rectified Linear Unit).
# Ela introduz não-linearidade no modelo, permitindo aprender padrões mais complexos.
model.add(tf.keras.layers.Activation('relu'))


# Adiciona uma camada de Max-Pooling.
# 'pool_size=(2, 2)' define o tamanho da janela de pooling. A camada reduz a dimensionalidade
# dos mapas de características, tornando o modelo mais eficiente e robusto a pequenas variações.
model.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2)))


# Adiciona uma camada Flatten, que transforma os mapas de características 2D em um vetor 1D.
# Isso é necessário para conectar as camadas convolucionais às camadas densas (totalmente conectadas).
model.add(tf.keras.layers.Flatten())


# Adiciona uma camada densa (totalmente conectada) com 100 neurônios (unidades).
model.add(tf.keras.layers.Dense(units=100))
# Adiciona novamente a função de ativação 'relu' após a camada densa.
model.add(tf.keras.layers.Activation('relu'))


# Adiciona a camada de saída, uma camada densa com 10 unidades, uma para cada classe (dígitos de 0 a 9).
model.add(tf.keras.layers.Dense(units=10))
# Adiciona a função de ativação 'softmax'. Ela converte as saídas em uma distribuição de
# probabilidade, onde cada valor representa a probabilidade de a imagem pertencer a uma classe.
model.add(tf.keras.layers.Activation('softmax'))


# Configura o otimizador, que é o algoritmo usado para ajustar os pesos do modelo.
# 'SGD' (Stochastic Gradient Descent) é usado aqui com parâmetros para melhorar a convergência:
# 'decay' para a taxa de decaimento do aprendizado, 'momentum' para acelerar o gradiente na direção correta,
# e 'nesterov=True' para usar o momentum de Nesterov.
optimizer = tf.keras.optimizers.SGD(decay=1e-6, momentum=0.9, nesterov=True)


# Compila o modelo, configurando o processo de aprendizado.
model.compile(
    # 'loss' define a função de perda. 'categorical_crossentropy' é usada para classificação
    # multi-classe com rótulos em formato one-hot.
    loss='categorical_crossentropy',
    # 'optimizer' define a regra de aprendizado (o otimizador configurado acima).
    optimizer=optimizer,
    # 'metrics' especifica as métricas a serem monitoradas durante o treinamento.
    metrics=['accuracy']
)


# Imprime um resumo da arquitetura do modelo.
# O resumo inclui a lista de camadas, o formato da saída de cada camada e o número de parâmetros treináveis.
print(model.summary())


Note the use of `Flatten()` to transform the 13x13x16 tensor representing the max-pooling layer into a vector of 2704 elements.

Note also the use of `padding='valid'` and its effect on the size of the tensor corresponding to the convolutional layer. Change it to `padding='same'` and conclude on its effect.

**Q:** Which layer has the most parameters? Why? Compare with the fully-connected MLPs you obtained during exercise 5.

**A:** conv layers have relatively few parameters (10 per filter). The main bottleneck is when going from convolutions to fully-connected.

Let's now train this network on MNIST for 10 epochs, using minibatches of 64 images:

In [ ]:
# History tracks the evolution of the metrics during learning
history = tf.keras.callbacks.History()

# Training procedure
model.fit(
    X_train, T_train, # training data
    batch_size=64,  # batch size
    epochs=10, # Maximum number of epochs
    validation_split=0.1, # Perceptage of training data used for validation
    callbacks=[history] # Track the metrics at the end of each epoch
)

As in the previous exercise, the next cells compute the test loss and accuracy and display the evolution of the training and validation accuracies:

In [ ]:
score = model.evaluate(X_test, T_test, verbose=0)
print('Test loss:', score[0])
print('Test accuracy:', score[1])

In [ ]:
plt.figure(figsize=(15, 6))

plt.subplot(121)
plt.plot(history.history['loss'], '-r', label="Training")
plt.plot(history.history['val_loss'], '-b', label="Validation")
plt.xlabel('Epoch #')
plt.ylabel('Loss')
plt.legend()

plt.subplot(122)
plt.plot(history.history['accuracy'], '-r', label="Training")
plt.plot(history.history['val_accuracy'], '-b', label="Validation")
plt.xlabel('Epoch #')
plt.ylabel('Accuracy')
plt.legend()

plt.show()

1. O que você acha da acurácia final e do tempo de treinamento, em comparação com o MLP da última vez?

Acurácia Final: A acurácia final é excelente. Olhando o gráfico da direita ("Accuracy"), a acurácia de validação (linha azul) se estabiliza em torno de 98.5%, enquanto a acurácia de treinamento (linha vermelha) se aproxima de 100%.

Uma acurácia de validação acima de 98% para o dataset MNIST é um resultado muito bom. Sem os gráficos do MLP para uma comparação direta, é provável que este modelo (uma Rede Neural Convolucional - CNN) tenha alcançado uma acurácia superior, já que CNNs são especializadas em tarefas de visão computacional.

Tempo de Treinamento: O tempo de treinamento não pode ser medido diretamente pelos gráficos, pois eles mostram o desempenho por época, não por tempo. No entanto, podemos inferir o seguinte:

As CNNs geralmente exigem mais computação por época do que um MLP simples, devido às operações de convolução. Portanto, cada época individualmente pode ter demorado mais.

Por outro lado, a CNN atinge uma alta acurácia muito rapidamente (acima de 98% em apenas 3-4 épocas), o que sugere que ela converge para uma boa solução mais rápido em termos de épocas do que um MLP normalmente faria.

2. Quando sua rede começa a apresentar overfitting? Como reconhecer?

O overfitting ocorre quando o modelo aprende o ruído e os detalhes específicos do conjunto de treinamento, perdendo a capacidade de generalizar para novos dados (o conjunto de validação). Nos gráficos, isso é identificado quando as curvas de treinamento e validação começam a divergir:



1.   A perda de treinamento (Training Loss) continua a diminuir.
2.   A perda de validação (Validation Loss) para de diminuir e começa a aumentar ou se estabiliza.
3. A acurácia de treinamento (Training Accuracy) continua a aumentar.
4. A acurácia de validação (Validation Accuracy) para de aumentar ou começa a cair

Observando os gráficos, o modelo começa a mostrar sinais de overfitting por volta da época 4.

No gráfico de "Loss", a perda de validação (azul) atinge seu ponto mais baixo por volta da época 4 e depois começa a subir lentamente, enquanto a perda de treinamento (vermelha) continua caindo.

No gráfico de "Accuracy", a acurácia de validação (azul) parece estagnar ou até cair ligeiramente após a época 4, enquanto a acurácia de treinamento (vermelha) continua a subir.

3. Tente valores diferentes para o tamanho do lote (batch size: 16, 32, 64, 128...). Qual é a sua influência?
O batch_size (tamanho do lote) é um hiperparâmetro que influencia a velocidade do treinamento, o uso de memória e a qualidade do modelo final.

**Tamanhos de Lote Menores (ex: 16, 32):**

Influência: As atualizações dos pesos do modelo são mais frequentes e "ruidosas" (menos estáveis).

Vantagens: Esse ruído pode ajudar o modelo a escapar de mínimos locais ruins e, muitas vezes, leva a uma melhor generalização (menor overfitting).

Desvantagens: O treinamento é mais lento porque o hardware (GPU/CPU) não é utilizado com eficiência máxima e mais atualizações são necessárias por época.

**Tamanhos de Lote Maiores (ex: 64, 128):**

Influência: As atualizações dos pesos são menos frequentes e baseadas em uma estimativa mais estável do gradiente.

Vantagens: O treinamento é mais rápido em termos de tempo por época, pois aproveita melhor a paralelização do hardware.

Desvantagens: Pode levar a uma generalização pior, pois o modelo tende a convergir para "mínimos locais agudos" ("sharp minima"), que podem não performar bem em dados não vistos.

**A:** A CNN, even as shallow as this one, is more accurate but much slower (on CPU) than fully-connected networks. A network overfits when the training accuracy becomes better than the validation accuracy (learning by heart, not generalizing), which is the case here. When the batch size is too small, learning is unstable: the training loss increases again after a while. 128 is actually slightly better than 64.

**Q:** Improve the CNN to avoid overfitting. The test accuracy should be around 99%.

You can:

* change the learning rate
* add another block on convolution + max-pooling before the fully-connected layer to reduce the number of parameters,
* add dropout after some of the layers,
* use L2 regularization,
* use a different optimizer,
* do whatever you want.

# Proposta de alteração 1 - Aplicando regularização L2

In [ ]:
# Importa o TensorFlow e o regularizador L2
from tensorflow.keras.regularizers import l2

# Limpa a sessão para liberar memória de modelos anteriores
tf.keras.backend.clear_session()

# Cria um modelo Sequencial
model = tf.keras.Sequential()

# Camada de entrada
model.add(tf.keras.layers.Input((28, 28, 1)))

# Primeira camada convolucional com regularização L2
# O argumento kernel_regularizer=l2(0.001) adiciona a penalidade aos pesos do kernel.
model.add(tf.keras.layers.Conv2D(32, (3, 3), activation='relu', padding='valid', kernel_regularizer=l2(0.001)))
model.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2)))
model.add(tf.keras.layers.Dropout(0.5))

# Segunda camada convolucional com regularização L2
model.add(tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='valid', kernel_regularizer=l2(0.001)))
model.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2)))
model.add(tf.keras.layers.Dropout(0.5))

# Camada Flatten
model.add(tf.keras.layers.Flatten())

# Camada densa com regularização L2
model.add(tf.keras.layers.Dense(150, activation='relu', kernel_regularizer=l2(0.001)))
model.add(tf.keras.layers.Dropout(0.5))

# Camada de saída (geralmente não se aplica regularização L2 na camada de saída)
model.add(tf.keras.layers.Dense(10, activation='softmax'))

# Otimizador (sem alterações)
optimizer = tf.keras.optimizers.SGD(decay=1e-6, momentum=0.9, nesterov=True)

# Compilação do modelo (sem alterações)
model.compile(
    loss='categorical_crossentropy',
    optimizer=optimizer,
    metrics=['accuracy']
)

# Imprime o resumo do modelo
print(model.summary())


In [ ]:
history = tf.keras.callbacks.History()

model.fit(
    X_train, T_train,
    batch_size=64,
    epochs=20,
    validation_split=0.1,
    callbacks=[history]
)

In [ ]:
score = model.evaluate(X_test, T_test, verbose=0)
print('Test loss:', score[0])
print('Test accuracy:', score[1])

In [ ]:
plt.figure(figsize=(15, 6))

plt.subplot(121)
plt.plot(history.history['loss'], '-r', label="Training")
plt.plot(history.history['val_loss'], '-b', label="Validation")
plt.xlabel('Epoch #')
plt.ylabel('Loss')
plt.legend()

plt.subplot(122)
plt.plot(history.history['accuracy'], '-r', label="Training")
plt.plot(history.history['val_accuracy'], '-b', label="Validation")
plt.xlabel('Epoch #')
plt.ylabel('Accuracy')
plt.legend()

plt.show()

1. O que você acha da acurácia final e do tempo de treinamento?

 A acurácia final deste modelo é boa. A acurácia de validação (linha azul) atinge um pico e se estabiliza em torno de 98.5%. Este resultado é comparável ao do modelo anterior, mas a principal melhoria não está no valor final, e sim no comportamento do modelo. A distância entre a acurácia de treinamento e a de validação é muito menor, e a perda de validação continua diminuindo junto com a de treinamento, o que indica um modelo mais robusto e que generaliza melhor.

O modelo atinge sua performance máxima de validação rapidamente, por volta da 5ª a 7ª época, e a mantém estável até o final. Isso mostra que o modelo converge de forma eficiente. Embora tenha sido treinado por 20 épocas, o treinamento poderia ter sido interrompido mais cedo (usando uma técnica como Early Stopping) sem perda de desempenho no conjunto de validação.

2. Quando sua rede começa a apresentar overfitting? Como reconhecer?

Com base nestes gráficos, a rede não apresenta sinais claros de overfitting ao longo das 20 épocas.

O principal indicador de overfitting seria a divergência das curvas: a perda de validação (azul) começaria a subir enquanto a perda de treinamento (vermelha) continuaria caindo. Isso não acontece aqui. Ambas as curvas de perda diminuem de forma paralela.

O fato de a perda de validação não aumentar e a acurácia de validação se manter estável em seu pico demonstra que as técnicas de regularização (Dropout e L2) foram bem-sucedidas em combater o overfitting. O modelo aprendeu a generalizar em vez de memorizar os dados de treino.


# Proposta 2 - Aumentando o Kernel

In [ ]:
# Limpa a sessão backend do Keras.
tf.keras.backend.clear_session()

# Inicia o modelo Sequencial
model = tf.keras.models.Sequential()

# Camada de entrada
model.add(tf.keras.layers.Input(shape=(28, 28, 1)))

# A primeira camada convolucional agora usa um kernel de tamanho (5, 5).
# Isso fará com que o filtro analise uma área de 5x5 pixels de cada vez.
model.add(tf.keras.layers.Conv2D(16, (5, 5), padding='valid'))
model.add(tf.keras.layers.Activation('relu'))

# Camada de Max-Pooling (sem alteração)
model.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2)))

# Camada Flatten (sem alteração)
model.add(tf.keras.layers.Flatten())

# Primeira camada densa (sem alteração)
model.add(tf.keras.layers.Dense(units=100))
model.add(tf.keras.layers.Activation('relu'))

# Camada de saída (sem alteração)
model.add(tf.keras.layers.Dense(units=10))
model.add(tf.keras.layers.Activation('softmax'))

# Otimizador (sem alteração)
optimizer = tf.keras.optimizers.SGD(decay=1e-6, momentum=0.9, nesterov=True)

# Compilação do modelo (sem alteração)
model.compile(
    loss='categorical_crossentropy',
    optimizer=optimizer,
    metrics=['accuracy']
)

# Imprime o resumo da nova arquitetura
print(model.summary())

In [ ]:
history = tf.keras.callbacks.History()

model.fit(
    X_train, T_train,
    batch_size=64,
    epochs=20,
    validation_split=0.1,
    callbacks=[history]
)

In [ ]:
score = model.evaluate(X_test, T_test, verbose=0)
print('Test loss:', score[0])
print('Test accuracy:', score[1])

In [ ]:
plt.figure(figsize=(15, 6))

plt.subplot(121)
plt.plot(history.history['loss'], '-r', label="Training")
plt.plot(history.history['val_loss'], '-b', label="Validation")
plt.xlabel('Epoch #')
plt.ylabel('Loss')
plt.legend()

plt.subplot(122)
plt.plot(history.history['accuracy'], '-r', label="Training")
plt.plot(history.history['val_accuracy'], '-b', label="Validation")
plt.xlabel('Epoch #')
plt.ylabel('Accuracy')
plt.legend()

plt.show()

1. O que você acha da acurácia final e do tempo de treinamento?

A acurácia final é boa. O modelo atinge uma acurácia de validação (linha azul) de aproximadamente 99%.

 O modelo é eficiente. Ele atinge sua performance máxima (próximo de 99%) em apenas 4 ou 5 épocas. Após esse ponto, a acurácia de validação não melhora significativamente. Isso sugere que o treinamento poderia ser interrompido bem mais cedo.

2. Quando sua rede começa a apresentar overfitting? Como reconhecer?
Análise: Este modelo apresenta sinais muito leves de overfitting, mas é extremamente bem controlado.

Observamos o início do overfitting no ponto em que as curvas de treinamento e validação começam a se afastar. No gráfico de "Loss", a perda de treinamento (vermelha) continua caindo de forma constante, enquanto a perda de validação (azul) atinge seu ponto mais baixo por volta da época 4 e depois se estabiliza, sem subir.

Embora o modelo continue a se especializar nos dados de treino (a perda de treino se aproxima de zero), ele não "desaprende" a generalizar. O fato de a perda de validação permanecer baixa e estável, e a acurácia de validação se manter no pico.

**Proposta 3 - Aumento no número de camadas**

In [ ]:
# Limpa a sessão backend do Keras.
tf.keras.backend.clear_session()

# Cria um modelo Sequencial
model = tf.keras.models.Sequential()

# Adiciona a camada de entrada
model.add(tf.keras.layers.Input(shape=(28, 28, 1)))


# --- PRIMEIRO BLOCO CONVOLUCIONAL ---
# Primeira camada convolucional.
# O padding='same' é para evitar que a imagem diminua muito de tamanho rapidamente.
model.add(tf.keras.layers.Conv2D(32, (3, 3), activation='relu', padding='same'))
model.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2)))


# --- SEGUNDO BLOCO CONVOLUCIONAL (NOVA CAMADA) ---
# Adicionamos uma segunda camada Conv2D.
# Isso permite que a rede aprenda características mais complexas a partir das características
# mais simples aprendidas pela primeira camada.
model.add(tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same'))
model.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2)))

model.add(tf.keras.layers.Flatten())

model.add(tf.keras.layers.Dense(units=128, activation='relu'))

model.add(tf.keras.layers.Dense(units=10, activation='softmax'))

optimizer = tf.keras.optimizers.SGD(decay=1e-6, momentum=0.9, nesterov=True)


# Compila o modelo
model.compile(
    loss='categorical_crossentropy',
    optimizer=optimizer,
    metrics=['accuracy']
)


# Imprime o resumo da nova arquitetura mais profunda
print(model.summary())


In [ ]:
history = tf.keras.callbacks.History()

model.fit(
    X_train, T_train,
    batch_size=64,
    epochs=20,
    validation_split=0.1,
    callbacks=[history]
)

In [ ]:
score = model.evaluate(X_test, T_test, verbose=0)
print('Test loss:', score[0])
print('Test accuracy:', score[1])

In [ ]:
plt.figure(figsize=(15, 6))

plt.subplot(121)
plt.plot(history.history['loss'], '-r', label="Training")
plt.plot(history.history['val_loss'], '-b', label="Validation")
plt.xlabel('Epoch #')
plt.ylabel('Loss')
plt.legend()

plt.subplot(122)
plt.plot(history.history['accuracy'], '-r', label="Training")
plt.plot(history.history['val_accuracy'], '-b', label="Validation")
plt.xlabel('Epoch #')
plt.ylabel('Accuracy')
plt.legend()

plt.show()

1. O que você acha da acurácia final e do tempo de treinamento?

A acurácia final é a melhor obtida. A acurácia de validação (linha azul) atinge e se estabiliza em um patamar impressionante, em torno de 99.2%. Este é um bom desempenho para o dataset MNIST e mostra que a camada adicional permitiu ao modelo aprender características mais ricas e discriminativas.

A convergência do modelo é  rápida. Ele atinge mais de 99% de acurácia de validação por volta da 4ª ou 5ª época. Após esse ponto, os ganhos são marginais. Isso indica que, apesar de ser mais profundo, o modelo é muito eficiente em aprender a tarefa.

2. Quando sua rede começa a apresentar overfitting? Como reconhecer?

O modelo começa a mostrar sinais de overfitting por volta da 3ª época.
 É neste ponto que as curvas de treinamento e validação começam a se separar claramente:

No gráfico de "Loss", a perda de validação (azul) para de cair e se estabiliza em um platô baixo. Em contraste, a perda de treinamento (vermelha) continua a diminuir de forma acentuada, aproximando-se de zero.

No gráfico de "Accuracy", a acurácia de validação atinge seu platô, enquanto a de treinamento continua subindo em direção a 100%.

O fato de o modelo continuar melhorando nos dados de treino, mas não nos de validação, indica que ele está começando a memorizar os exemplos de treinamento. No entanto, o overfitting é bem controlado, pois a acurácia de validação não cai; ela apenas para de melhorar.

## Analysing the CNN

Once a network has been trained, let's see what has happened internally.

### Accessing trained weights

Each layer of the network can be addressed individually. For example, `model.layers[0]` represents the first layer of your network (the first convolutional one, as the input layer does not count). The index of the other layers can be found by looking at the output of `model.summary()`.

You can obtain the parameters of each layer (if any) with:

```python
W = model.layers[0].get_weights()[0]
```

**Q:** Print the shape of these weights and relate them to the network.

In [ ]:
W = model.layers[0].get_weights()[0]
print("W shape : ", W.shape)

**Q:** Visualize with `imshow()` each of the 16 filters of the first convolutional layer. Interpret what kind of operation they perform on the image.

*Hint:* `subplot()` is going to be useful here. If you have 16 images `img[i]`, you can visualize them in a 4x4 grid with:

```python
for i in range(16):
    plt.subplot(4, 4, i+1)
    plt.imshow(img[i], cmap=plt.cm.gray)
```

In [ ]:
plt.figure(figsize=(12, 12))
for i in range(16):
    plt.subplot(4, 4, i+1)
    plt.imshow(W[:, :, 0, i], cmap=plt.cm.gray)
    plt.xticks([]); plt.yticks([])
    plt.colorbar()
plt.show()

### Visualizing the feature maps

Let's take a random image from the training set and visualize it:

In [ ]:
idx = 31727 # or any other digit
x = X_train[idx, :, :, :].reshape(1, 28, 28, 1)
t = t_train[idx]

print(t)

plt.figure(figsize=(6, 6))
plt.imshow(x[0, :, :, 0] + X_mean[:, :, 0], cmap=plt.cm.gray)
plt.colorbar()
plt.show()

This example could be a 1 or 7. That is why you will never get 100% accuracy on MNIST: some examples are hard even for humans...

**Q:** Print what the model predict for it, its true label, and visualize the probabilities in the softmax output layer (look at the doc of `model.predict()`):

In [ ]:
# Predict probabilities
output = model.predict([x])

# The predicted class has the maximal probability
prediction = output[0].argmax()
print('Predicted digit:', prediction, '; True digit:', t)

plt.figure(figsize=(12, 5))
plt.bar(range(10), output[0])
plt.xlabel('Digit')
plt.ylabel('Probability')
plt.show()

Depending on how your network converged, you may have the correct prediction or not.

**Q:** Visualize the output of the network for different examples. Do these ambiguities happen often?

Now let's look inside the network. We will first visualize the 16 feature maps of the first convolutional layer.

This is actually very simple using tensorflow 2.x: One only needs to create a new model (class `tf.keras.models.Model`, not Sequential) taking the same inputs as the original model, but returning the output of the first layer (`model.layers[0]` is the first convolutional layer of the model, as the input layer does not count):

```python
model_conv = tf.keras.models.Model(inputs=model.inputs, outputs=model.layers[0].output)
```

To get the tensor corresponding to the first convolutional layer, one simply needs to call `predict()` on the new model:

```python
feature_maps = model_conv.predict([x])
```

**Q:** Visualize the 16 feature maps using `subplot()`. Relate these activation with the filters you have visualized previously.

In [ ]:
model_conv = tf.keras.models.Model(inputs=model.inputs, outputs=model.layers[0].output)

feature_maps = model_conv.predict([x])
print(feature_maps.shape)

plt.figure(figsize=(12, 12))
for i in range(16):
    plt.subplot(4, 4, i+1)
    plt.imshow(feature_maps[0, :, :, i], cmap=plt.cm.gray)
    plt.xticks([]); plt.yticks([])
    plt.colorbar()
plt.show()

**Q:** Do the same with the output of the first max-pooling layer.

*Hint:* you need to find the index of that layer in `model.summary()`.

In [ ]:
model_pool = tf.keras.models.Model(inputs=model.inputs, outputs=model.layers[1].output)

pooling_maps = model_pool.predict([x])
print(pooling_maps.shape)

plt.figure(figsize=(12, 12))
for i in range(16):
    plt.subplot(4, 4, i+1)
    plt.imshow(pooling_maps[0, :, :, i], cmap=plt.cm.gray)
    plt.xticks([]); plt.yticks([])
    plt.colorbar()
plt.show()

**Bonus question:** if you had several convolutional layers in your network, visualize them too. What do you think of the specificity of some features?

In [ ]:
model_conv = tf.keras.models.Model(inputs=model.inputs, outputs=model.layers[3].output)

feature_maps = model_conv.predict([x])
print(feature_maps.shape)

plt.figure(figsize=(12, 12))
for i in range(16):
    plt.subplot(4, 4, i+1)
    plt.imshow(feature_maps[0, :, :, i], cmap=plt.cm.gray)
    plt.xticks([]); plt.yticks([])
    plt.colorbar()
plt.show()